# Confidence Analysis

**Purpose:** Test three hypotheses about how training shapes introspective confidence:
- **H1 (Better readout):** Internal uncertainty signal exists in base; training improves the path from signal to output token.
- **H2 (Stronger representation):** Training enhances the internal uncertainty representation itself.
- **H3 (Reporting policy):** Internal representation unchanged; only late-stage mapping to output changes.

**Scope:** Confidence task only (delegate task excluded). Single token position (last prompt token before generation).

**Models:** base, instruct, finetuned (Llama-3.1-8B family).  
**Datasets:** SimpleMC, TriviaMC.

**Build order (do not skip ahead):** Section 1 → sanity check → Section 2 → ... Each section must have a short interpretation cell before moving on.

---
## Section 0: Setup and hypothesis framing

In [ ]:
# --- Imports ---
from __future__ import annotations

import json
import pickle
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import RidgeCV, LogisticRegressionCV
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import spearmanr, pearsonr

# --- Globals ---
SEED = 42
np.random.seed(SEED)

N_SPLITS = 5
RIDGE_ALPHAS = np.logspace(-2, 4, 13)
LOGIT_C = np.logspace(-2, 2, 9)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# --- Paths ---
REPO_ROOT = Path('.').resolve()
OUTPUTS_DIR = REPO_ROOT / 'outputs'
CACHE_DIR = REPO_ROOT / 'outputs' / 'analysis_cache'
FIGURES_DIR = REPO_ROOT / 'outputs' / 'analysis_figures'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repo root: {REPO_ROOT}')
print(f'Outputs:   {OUTPUTS_DIR}')
print(f'Cache:     {CACHE_DIR}')
print(f'Figures:   {FIGURES_DIR}')

In [ ]:
# --- Run configuration ---
# Map each model label to its output folder. Update as new runs come in.
# Each folder should contain paired_data.json and activations.npz files
# following the pattern {model_short}_{dataset}_introspection_*.

MODELS = {
    'base':      {'folder': '8b_base_26_04',      'model_short': 'Llama-3.1-8B'},
    'instruct':  {'folder': '8b_instruct_26_04',  'model_short': 'Llama-3.1-8B-Instruct'},
    'finetuned': {'folder': '8b_finetuned_26_04', 'model_short': 'Llama-3.1-8B-Instruct'},
}

DATASETS = ['SimpleMC', 'TriviaMC']
META_TASK = 'confidence'  # delegate excluded from this notebook

# Column-name translation from Opus's brief to our data:
#   top2_margin_logit    -> 'logit_gap'     (z(top) - z(second))
#   entropy_over_options -> 'entropy'       (entropy of option distribution)
#   max_option_prob      -> 'top_prob'

# Probe targets (Sections 2+):
TARGETS = {
    'margin_logit':      {'col': 'logit_gap',                 'kind': 'regression'},   # primary
    'margin_prob':       {'col': 'margin',                    'kind': 'regression'},
    'entropy':           {'col': 'entropy',                   'kind': 'regression'},
    'stated_confidence': {'col': 'stated_confidence_numeric', 'kind': 'regression'},
    'is_correct':        {'col': 'is_correct',                'kind': 'classification'},
}

In [ ]:
# --- Data loading utilities ---

@dataclass
class RunData:
    """Container for one (model, dataset) run."""
    model: str                       # e.g. 'base'
    dataset: str                     # e.g. 'SimpleMC'
    df: pd.DataFrame                 # per-example metadata (one row per question)
    direct_acts: np.ndarray          # (n_examples, n_layers, d_model), float16, last-prompt token
    meta_acts: np.ndarray            # same shape, confidence-prompt last token
    config: dict                     # run config

    @property
    def n_examples(self) -> int:
        return self.direct_acts.shape[0]

    @property
    def n_layers(self) -> int:
        return self.direct_acts.shape[1]

    @property
    def d_model(self) -> int:
        return self.direct_acts.shape[2]


def _resolve_paths(model: str, dataset: str, meta_task: str = META_TASK) -> Dict[str, Path]:
    """Build expected file paths for (model, dataset). Handles meta_task suffix."""
    cfg = MODELS[model]
    folder = OUTPUTS_DIR / cfg['folder']
    ms = cfg['model_short']
    suffix = '' if meta_task == 'confidence' else f'_{meta_task}'
    stem = f'{ms}_{dataset}_introspection{suffix}'
    return {
        'folder': folder,
        'paired_data': folder / f'{stem}_paired_data.json',
        'direct_acts': folder / f'{stem}_direct_activations.npz',
        'meta_acts':   folder / f'{stem}_meta_activations.npz',
    }


def _stack_layer_npz(npz_path: Path) -> np.ndarray:
    """Load an activations npz with layer_0..layer_N keys into (n, n_layers, d)."""
    with np.load(npz_path) as f:
        layer_keys = sorted(
            [k for k in f.keys() if k.startswith('layer_')],
            key=lambda k: int(k.split('_')[1]),
        )
        stacked = np.stack([f[k] for k in layer_keys], axis=1)  # (n, n_layers, d)
    return stacked.astype(np.float16)


def _build_dataframe(paired: dict, model: str, dataset: str) -> pd.DataFrame:
    """Flatten paired_data.json per-example fields into a DataFrame.

    One row per question. Columns include all direct_metrics, stated_confidence_numeric,
    is_correct, direct_responses, meta_responses, and question metadata.
    """
    n = len(paired['direct_probs'])
    row = {
        'example_id':  [q.get('id', f'q_{i}') for i, q in enumerate(paired['questions'])],
        'model':       model,
        'dataset':     dataset,
        'question':    [q.get('question', '') for q in paired['questions']],
        'correct_answer': [q.get('correct_answer', '') for q in paired['questions']],
        'direct_response': paired.get('direct_responses', [None] * n),
        'meta_response':   paired.get('meta_responses',   [None] * n),
        'is_correct':      paired.get('is_correct', [False] * n),
        'stated_confidence_numeric': paired.get('stated_confidence_numeric', [np.nan] * n),
        'meta_entropy':    paired.get('meta_entropies', [np.nan] * n),
    }
    for k, v in paired.get('direct_metrics', {}).items():
        row[k] = v
    if 'other_confidence' in paired:
        row['other_confidence_signal'] = paired['other_confidence'].get('signals', [np.nan] * n)
    return pd.DataFrame(row).reset_index(drop=True)


def load_run(model: str, dataset: str, meta_task: str = META_TASK) -> RunData:
    """Load one (model, dataset) run. Activations stacked to float16 array."""
    paths = _resolve_paths(model, dataset, meta_task)
    if not paths['paired_data'].exists():
        raise FileNotFoundError(f'paired_data not found: {paths["paired_data"]}')
    with open(paths['paired_data']) as f:
        paired = json.load(f)
    direct_acts = _stack_layer_npz(paths['direct_acts'])
    meta_acts   = _stack_layer_npz(paths['meta_acts'])
    df = _build_dataframe(paired, model, dataset)
    assert len(df) == direct_acts.shape[0] == meta_acts.shape[0], (
        f'Row count mismatch: df={len(df)} direct={direct_acts.shape[0]} meta={meta_acts.shape[0]}'
    )
    return RunData(
        model=model, dataset=dataset, df=df,
        direct_acts=direct_acts, meta_acts=meta_acts,
        config=paired.get('config', {}),
    )


def load_all_available(meta_task: str = META_TASK) -> Dict[Tuple[str, str], RunData]:
    """Load every (model, dataset) combination that exists on disk; skip missing."""
    runs: Dict[Tuple[str, str], RunData] = {}
    for model in MODELS:
        for dataset in DATASETS:
            try:
                r = load_run(model, dataset, meta_task)
                runs[(model, dataset)] = r
                print(f'  loaded {model}/{dataset}: n={r.n_examples}, layers={r.n_layers}, d={r.d_model}')
            except FileNotFoundError as e:
                print(f'  skipped {model}/{dataset}: missing file')
    return runs

In [ ]:
# --- Load all available runs ---
RUNS = load_all_available()
print(f'\nTotal runs loaded: {len(RUNS)}')
print(f'Keys: {list(RUNS.keys())}')

In [ ]:
# --- Unified per-example DataFrame across all runs ---

def build_unified_df(runs: Dict[Tuple[str, str], RunData]) -> pd.DataFrame:
    return pd.concat([r.df for r in runs.values()], ignore_index=True)

UNIFIED_DF = build_unified_df(RUNS) if RUNS else pd.DataFrame()
print(f'Unified df shape: {UNIFIED_DF.shape}')
if len(UNIFIED_DF):
    print(UNIFIED_DF[['model', 'dataset']].value_counts())

### Hypothesis scorecard

Predicted signatures for each hypothesis. Section 10 fills in observed values and scores matches.

| Signature | H1 (Better readout) | H2 (Stronger repr) | H3 (Reporting policy) |
|---|---|---|---|
| **within_R2_margin** (base → finetuned) | ≈ unchanged (high in base already) | **increases** at matched layers | ≈ unchanged |
| **cross_R2 margin→stated** (base → finetuned) | **increases sharply** | increases | increases (late layers) |
| **ablation effect on stated confidence** | **large** | large | **large (late-layer only)** |
| **ablation effect on margin/entropy** | **small** | medium | **small** |
| **layer_localization of gains** | mid-to-late | **all layers** | **last 2–4 layers** |


In [ ]:
# --- Hypothesis scorecard DataFrame (filled in Section 10) ---
SCORECARD = pd.DataFrame(
    {
        'within_R2_margin':           ['unchanged (high in base)', 'increases',  'unchanged'],
        'cross_R2_margin_to_stated':  ['increases sharply',         'increases',  'increases (late)'],
        'ablation_effect_stated':     ['large',                     'large',      'large (late only)'],
        'ablation_effect_margin':     ['small',                     'medium',     'small'],
        'layer_localization':         ['mid-to-late',               'all layers', 'last 2–4 layers'],
    },
    index=['H1', 'H2', 'H3'],
)
print('\n=== Predicted signatures ===')
SCORECARD

---
## Section 1: Dataset characterization and sanity checks

**Purpose:** Verify the data before any mechanistic analysis.
- Entropy distributions (expect right-skew)
- Correlation among `entropy`, `logit_gap`, `top_prob` (expect ≥ 0.9 within-model)
- Reliability diagram (stated confidence vs. actual accuracy, one line per model)
- Sample counts per (model, dataset, entropy quartile) — flag cells with n < 200
- Define `entropy_balanced_subset(df)` helper

**Hypothesis relevance:** Prerequisite only. No hypothesis claims tested here. Pathological correlations or reliability curves are red flags for everything downstream.

**Flags to watch for:**
- Base model reliability should be roughly flat (base can't self-report calibration well) — monotone curve means few-shot prompt is leaking signal.
- If `entropy`–`logit_gap`–`top_prob` correlations are much below 0.9, metrics are diverging meaningfully.

In [ ]:
# Section 1 analysis goes here (to be filled after scaffolding approval).

In [ ]:
# Reusable helper: subsample to balance entropy quartiles.
# Placeholder — implementation in Section 1.

def entropy_balanced_subset(df: pd.DataFrame, entropy_col: str = 'entropy', seed: int = SEED) -> pd.DataFrame:
    """Return a subset with approximately equal counts across entropy quartiles.

    Quartiles are computed within the input df (not globally). The dominant
    (typically lowest-entropy) bin is subsampled down to the smallest bin's size.
    """
    raise NotImplementedError('Implemented in Section 1.')

**Section 1 summary (fill in after running):** _TBD_

---
## Section 2: Within-model linear probes

**Purpose:** For each (model, dataset, layer), fit ridge-regression (or logistic-regression for `is_correct`) probes on direct-prompt activations to predict:
- `margin_logit` (primary, linear)
- `margin_prob` (prob-space)
- `entropy`
- `stated_confidence_numeric`
- `is_correct` (classification)

Also fit mean-difference contrastive directions for top-quartile vs bottom-quartile entropy (discard middle 50% for fitting). Save probe weights, held-out projections, and direction vectors to `CACHE_DIR`.

**Evaluation:** 5-fold CV, held-out R² (accuracy for classification). Report on both natural and entropy-balanced subsets.

**Hypothesis relevance:**
- Margin R² **increases base→finetuned** at matched layers → **H2**.
- Margin R² already **high in base, unchanged** → **H1**.
- R² gains **concentrate in final layers** → **H3**.

**Key figure:** Layer-wise R² curves per (dataset, target), one line per model.

In [ ]:
# Section 2 analysis goes here.

**Section 2 summary:** _TBD_

---
## Section 3: Cross-prediction analysis

**Purpose:** For each model × layer: fit probe on target X, predict target Y on held-out data. Pairs among `{margin_logit, entropy, stated_confidence, is_correct}`.

Report cross-R² and the ratio `cross_R2 / mean(within_R2_X, within_R2_Y)`.

**Key figure:** Layer-wise cross-R² for `margin_logit → stated_confidence`, alongside each within-R². Three subplots, one per model. **Distinguishes H1 from H2/H3.**

**Hypothesis relevance:**
- **H1:** cross margin→stated increases sharply base→finetuned; within-margin roughly unchanged.
- **H2:** both within and cross increase.
- **H3:** gain concentrated in late layers.

In [ ]:
# Section 3 analysis goes here.

**Section 3 summary:** _TBD_

---
## Section 4: Cosine similarity and projection correlation (diagnostic)

**Purpose:** Diagnostic comparison between the "margin" direction and the "stated confidence" direction within each model, per layer.

- **Cosine:** geometric similarity of probe weight vectors (and mean-diff directions).
- **Projection correlation:** correlation across examples of scalar projections onto A vs B. Functional similarity independent of geometry.

**Flag:** Low cosine + high projection correlation → geometrically different but functionally equivalent directions.

In [ ]:
# Section 4 analysis goes here.

**Section 4 summary:** _TBD_

---
## Section 5: Cross-model comparison via behavioral alignment

**Purpose:** Activation bases aren't aligned across models. Compare via scalar projections.

For each model, take per-example projections onto its own fitted margin direction and its own fitted stated-confidence direction (from Section 2, at each model's peak layer).

**Plots:** Scatter of projection(A) vs projection(B) for each pair (base↔instruct, instruct↔finetuned, base↔finetuned). Spearman correlation per pair.

**Interpretation:** Strong base↔finetuned correlation → underlying item ordering preserved → precondition for claiming training operates on a shared signal.

In [ ]:
# Section 5 analysis goes here.

**Section 5 summary:** _TBD_

---
## Section 6: Representational similarity analysis (RSA)

**Purpose:** For each model × layer, compute pairwise cosine/correlation matrix over a fixed shared stimulus subset. Compare RSMs across models via Spearman correlation of off-diagonal entries.

**Visualization:** Heatmap of RSM similarity between models, layer by layer.

**Interpretation:**
- High at all layers → broadly shared representations, training is readout change (**H1/H3**).
- Drops at late layers → training operates on late representations (**H3**).

In [ ]:
# Section 6 analysis goes here.

**Section 6 summary:** _TBD_

---
## Section 7: Cross-decoding across models

**Purpose:** Extend cross-prediction across models. Fit probe on X in model A, apply to model B's activations to predict X.

Report held-out R² of cross-model probe vs within-model. Bases aren't aligned — point is **quantifying how much it fails.**

**Interpretation:**
- Cross-model probe works passably (within ~30% of within-model) → representation largely preserved.
- Collapses → finetuning substantially changed representation (favors **H2**).

In [ ]:
# Section 7 analysis goes here.

**Section 7 summary:** _TBD_

---
## Section 8: Incorporate ablation results (placeholder)

**Purpose:** Load pre-computed ablation results (path TBD), summarize effect on stated confidence vs. margin/entropy across models and layers. Compare against controls: random direction, orthogonal direction, matched noise, layer sweep.

**Hypothesis relevance:** Mechanistic claims rest here. Correlational analyses above are consistent with shared information but don't prove mechanism.
- **H1:** ablating stated-confidence direction → large effect on stated, small on margin.
- **H2:** ablation hurts underlying margin somewhat.
- **H3:** late-layer ablations large; mid-layer small.

In [ ]:
# Section 8 analysis placeholder.

**Section 8 summary:** _TBD (awaiting ablation results file path)_

---
## Section 9: Persona extension (stub)

**Purpose:** If persona-conditioned runs exist, fit margin and stated-confidence probes per persona; test cross-persona transfer.

**Prediction:** If personas share an uncertainty computation but differ in reporting, margin direction should be more shared across personas than stated-confidence direction.

**Status:** No persona data currently collected. Stub only.

In [ ]:
# Section 9 — stub. Skip until persona data is available.

---
## Section 10: Fill in the hypothesis scorecard

**Purpose:** Return to the scorecard from Section 0. For each cell (hypothesis × signature), fill in whether the empirical result matches, partially matches, or contradicts the prediction. Write a paragraph summary.

In [ ]:
# Section 10 scorecard fill-in.

**Section 10 summary:** _TBD_

---
## Section 11: Export key figures and summary tables

- Save main figures to `FIGURES_DIR` as PDFs.
- Save summary CSV of key metrics (within-R², cross-R², projection correlations, ablation effects) indexed by (model, dataset, layer, target) to `CACHE_DIR`.

In [ ]:
# Section 11 export.